### 手动实现GoogLeNet（Inception v1），并测试模块功能


In [1]:
import torch
import torch.nn as nn
from inception_module import InceptionModule, GoogLeNet

#### 1. 测试单个Inception模块


In [2]:
# 创建一个Inception模块（以Inception 3a为例）
# 输入: 192通道, 输出: 64+128+32+32 = 256通道
inception_3a = InceptionModule(
    in_channels=192,
    n1x1=64,
    n3x3_reduce=96,
    n3x3=128,
    n5x5_reduce=16,
    n5x5=32,
    pool_proj=32
)

print("Inception 3a 模块结构:")
print(inception_3a)
print("\n" + "="*50 + "\n")

# 测试前向传播
x = torch.randn(1, 192, 28, 28)  # batch_size=1, channels=192, height=28, width=28
output = inception_3a(x)
print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"输出通道数: {output.shape[1]} (预期： 64+128+32+32=256)")

Inception 3a 模块结构:
InceptionModule(
  (branch1): Sequential(
    (0): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU(inplace=True)
  )
  (branch2): Sequential(
    (0): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
  )
  (branch3): Sequential(
    (0): Conv2d(192, 16, kernel_size=(1, 1), stride=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (3): ReLU(inplace=True)
  )
  (branch4): Sequential(
    (0): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
    (1): Conv2d(192, 32, kernel_size=(1, 1), stride=(1, 1))
    (2): ReLU(inplace=True)
  )
)


输入形状: torch.Size([1, 192, 28, 28])
输出形状: torch.Size([1, 256, 28, 28])
输出通道数: 256 (预期： 64+128+32+32=256)


#### 2. 完整GoogLeNet网络

In [3]:
model = GoogLeNet(num_classes=1000, aux_classifiers=True)
print(model)

GoogLeNet(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception3a): InceptionModule(
    (branch1): Sequential(
      (0): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=True)
    )
    (branch2): Sequential(
      (0): Conv2d(192, 96, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(96, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
    )
    (branch3): Sequential(
      (0): Conv2d(192, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(16, 32, kernel_size=(5, 5), stri

#### 3. 前向传播

手动模型定义中，通过识别模型`self.training`状态确定辅助分类器是否进行输出

In [4]:
x = torch.randn(2, 3, 224, 224)  # batch_size=2, channels=3, height=224, width=224

model.train()
main_out, aux1_out, aux2_out = model(x)
print("训练模式输出:")
print(f"主分类器输出形状: {main_out.shape}")
print(f"辅助分类器1输出形状: {aux1_out.shape}")
print(f"辅助分类器2输出形状: {aux2_out.shape}")

print("\n" + "="*50 + "\n")

# 推理模式：仅返回主输出
model.eval()
with torch.no_grad():
    output = model(x)
print("推理模式输出:")
print(f"输出形状: {output.shape}")
print(f"预测类别: {output.argmax(dim=1)}")


训练模式输出:
主分类器输出形状: torch.Size([2, 1000])
辅助分类器1输出形状: torch.Size([2, 1000])
辅助分类器2输出形状: torch.Size([2, 1000])


推理模式输出:
输出形状: torch.Size([2, 1000])
预测类别: tensor([494, 494])


#### 4.与Torchvision版本对比

In [5]:
# 导入torchvision的GoogLeNet
import torchvision.models as models

# 加载预训练的GoogLeNet
googlenet_official = models.googlenet(weights=models.GoogLeNet_Weights.IMAGENET1K_V1)
print("Torchvision官方GoogLeNet结构:")
print(googlenet_official)

Torchvision官方GoogLeNet结构:
GoogLeNet(
  (conv1): BasicConv2d(
    (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (conv2): BasicConv2d(
    (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (conv3): BasicConv2d(
    (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=True)
  (inception3a): Inception(
    (branch1): BasicConv2d(
      (conv): Conv2d(192, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, moment

In [6]:
# 比较参数量
our_params = sum(p.numel() for p in model.parameters())
official_params = sum(p.numel() for p in googlenet_official.parameters())

print("="*60)
print("参数量对比")
print("="*60)
print(f"手动实现的GoogLeNet: {our_params:,}")
print(f"Torchvision官方版本: {official_params:,}")

参数量对比
手动实现的GoogLeNet: 13,374,120
Torchvision官方版本: 6,624,904


#### 5 参数差异分析
手动实现的版本是Inception v1，Torchvision的版本是**v2及之后更新**的，主要区别是：
* **用2个3x3卷积代替5x5卷积**
* **使用BatchNorm**
* **禁用bias**

因此参数量减少很多。